# NB4 — Hierarchical Cascade Classifier

Trains a two-stage cascade:
- **Stage 1**: Random Forest — Normal vs Anomaly (attack + fault)
- **Stage 2**: Random Forest — Attack vs Fault (anomaly rows only)

Evaluates Stage 2 standalone (upper bound), full cascade performance, per-fault-type recall, and FPR budget sensitivity. Saves models, tables, and figures.

In [1]:
from __future__ import annotations
from datetime import datetime
from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report

pd.set_option("display.max_columns", 40)

DATA_DIR    = Path("data")
MODELS_DIR  = DATA_DIR / "models"
RESULTS_DIR = DATA_DIR / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
for d in [MODELS_DIR, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RF_N_ESTIMATORS = 100
RF_RANDOM_STATE = 42
FPR_BUDGETS     = [0.005, 0.010, 0.020]  # 0.5%, 1.0%, 2.0%

# Load features
df = pd.read_parquet(DATA_DIR / "features.parquet")
feat_ref   = json.loads((DATA_DIR / "feature_cols.json").read_text())
FEATURE_COLS = feat_ref["feature_cols"]

print(f"Loaded: {df.shape}")
print(f"Feature columns: {len(FEATURE_COLS)}")
print(f"\nLabel counts:")
for lv, ln in [(0,"normal"),(1,"attack"),(2,"fault")]:
    print(f"  {ln} ({lv}): {(df['label']==lv).sum():>9,}")


Loaded: (806852, 814)
Feature columns: 810

Label counts:
  normal (0):   786,402
  attack (1):     9,977
  fault (2):    10,473


## 1. Train/Test Split

In [2]:
train_df = df[df["split"] == "train"].copy()
test_df  = df[df["split"] == "test"].copy()

X_train = train_df[FEATURE_COLS].values
X_test  = test_df[FEATURE_COLS].values

# Stage 1 labels: 0=normal, 1=anomaly (attack or fault)
y_train_s1 = (train_df["label"] > 0).astype(int).values
y_test_s1  = (test_df["label"] > 0).astype(int).values

# Original 3-class test labels
y_test_orig = test_df["label"].values
fault_type_test = test_df["fault_type"].values if "fault_type" in test_df.columns else None

print(f"Train rows: {len(train_df):,}  (anomaly={y_train_s1.sum():,}  normal={(~y_train_s1.astype(bool)).sum():,})")
print(f"Test rows:  {len(test_df):,}  (normal={(y_test_orig==0).sum():,}  attack={(y_test_orig==1).sum():,}  fault={(y_test_orig==2).sum():,})")


Train rows: 636,979  (anomaly=16,563  normal=620,416)
Test rows:  169,873  (normal=165,986  attack=1,953  fault=1,934)


## 2. Stage 1 — Normal vs Anomaly RF

This can take ~ 30 minutes

In [3]:
print("Training Stage 1 RF (normal vs anomaly)...")
rf_s1 = RandomForestClassifier(
    n_estimators=RF_N_ESTIMATORS,
    random_state=RF_RANDOM_STATE,
    oob_score=True,
    n_jobs=-1,
)
rf_s1.fit(X_train, y_train_s1)
print(f"Stage 1 OOB score: {rf_s1.oob_score_:.4f}")

p_s1 = rf_s1.predict_proba(X_test)[:, 1]  # P(anomaly)


Training Stage 1 RF (normal vs anomaly)...
Stage 1 OOB score: 0.9974


## 3. Stage 1 Threshold Sweep — Operating Points

In [4]:
n_normal_test  = (y_test_orig == 0).sum()
n_attack_test  = (y_test_orig == 1).sum()
n_fault_test   = (y_test_orig == 2).sum()

sweep_thresholds = np.linspace(0.01, 0.99, 500)
results_s1 = []

for t in sweep_thresholds:
    pred_anomaly = p_s1 >= t
    fp     = ((y_test_orig == 0) & pred_anomaly).sum()
    fpr    = fp / n_normal_test if n_normal_test > 0 else 0
    atk_r  = ((y_test_orig == 1) & pred_anomaly).sum() / n_attack_test if n_attack_test > 0 else 0
    flt_r  = ((y_test_orig == 2) & pred_anomaly).sum() / n_fault_test  if n_fault_test  > 0 else 0
    results_s1.append({"threshold": t, "fpr": fpr, "attack_recall": atk_r, "fault_recall": flt_r})

s1_df = pd.DataFrame(results_s1)

print("Stage 1 Operating Points:")
print(f"{'FPR Budget':>12}  {'Threshold':>10}  {'Fault Recall':>13}  {'Attack Recall':>14}  {'Normal FPR':>11}")
operating_points = {}
for budget in FPR_BUDGETS:
    feasible = s1_df[s1_df["fpr"] <= budget]
    if feasible.empty:
        print(f"  {budget:.1%}  — no feasible threshold")
        continue
    row = feasible.loc[feasible["fault_recall"].idxmax()]
    operating_points[budget] = row
    print(f"  {budget:.1%}  t={row['threshold']:.3f}  fault_recall={row['fault_recall']:.1%}  "
          f"attack_recall={row['attack_recall']:.1%}  fpr={row['fpr']:.2%}")


Stage 1 Operating Points:
  FPR Budget   Threshold   Fault Recall   Attack Recall   Normal FPR
  0.5%  t=0.440  fault_recall=26.9%  attack_recall=99.3%  fpr=0.49%
  1.0%  t=0.320  fault_recall=40.7%  attack_recall=99.7%  fpr=0.99%
  2.0%  t=0.201  fault_recall=51.8%  attack_recall=99.9%  fpr=1.88%


## 4. Stage 2 — Attack vs Fault RF (trained on anomaly rows only)

In [5]:
print("Training Stage 2 RF (attack vs fault, anomaly rows only)...")
anomaly_mask_train = train_df["label"].isin([1, 2])
train_s2_df = train_df[anomaly_mask_train].copy()
X_train_s2  = train_s2_df[FEATURE_COLS].values
y_train_s2  = (train_s2_df["label"] == 1).astype(int).values  # 1=attack, 0=fault

rf_s2 = RandomForestClassifier(
    n_estimators=RF_N_ESTIMATORS,
    random_state=RF_RANDOM_STATE,
    oob_score=True,
    n_jobs=-1,
)
rf_s2.fit(X_train_s2, y_train_s2)
print(f"Stage 2 OOB score: {rf_s2.oob_score_:.4f}")
print(f"Training set: {len(train_s2_df):,} anomaly rows  "
      f"(attack={y_train_s2.sum():,}  fault={(~y_train_s2.astype(bool)).sum():,})")

p_s2 = rf_s2.predict_proba(X_test)[:, 1]  # P(attack | row)


Training Stage 2 RF (attack vs fault, anomaly rows only)...
Stage 2 OOB score: 0.9976
Training set: 16,563 anomaly rows  (attack=8,024  fault=8,539)


## 5. Stage 2 Standalone Evaluation (Upper Bound)

Evaluates Stage 2 on clean anomaly-only test rows — this shows how well the attack/fault discriminator performs when given perfect anomaly detection. Demonstrates that Stage 1 fault recall is the bottleneck.

In [6]:
anomaly_test_mask = y_test_orig > 0
y_test_anomaly    = y_test_orig[anomaly_test_mask]  # 1=attack, 2=fault
p_s2_anomaly      = p_s2[anomaly_test_mask]

# At threshold 0.5 for attack
s2_pred = np.where(p_s2_anomaly >= 0.5, 1, 2)
s2_attack_recall = (s2_pred[y_test_anomaly == 1] == 1).mean()
s2_fault_recall  = (s2_pred[y_test_anomaly == 2] == 2).mean()
s2_accuracy      = (s2_pred == y_test_anomaly).mean()

print("Stage 2 Standalone (on anomaly-only test rows):")
print(f"  Attack recall: {s2_attack_recall:.1%}")
print(f"  Fault recall:  {s2_fault_recall:.1%}")
print(f"  Accuracy:      {s2_accuracy:.1%}")
print(f"  N anomaly rows: {anomaly_test_mask.sum():,}  "
      f"(attack={( y_test_anomaly==1).sum():,}  fault={(y_test_anomaly==2).sum():,})")
print("\nConclusion: Stage 2 discriminates well. Stage 1 fault recall is the bottleneck.")


Stage 2 Standalone (on anomaly-only test rows):
  Attack recall: 100.0%
  Fault recall:  99.0%
  Accuracy:      99.5%
  N anomaly rows: 3,887  (attack=1,953  fault=1,934)

Conclusion: Stage 2 discriminates well. Stage 1 fault recall is the bottleneck.


## 6. Full Cascade Evaluation

In [7]:
cascade_results = []

for budget in FPR_BUDGETS:
    if budget not in operating_points:
        continue
    row = operating_points[budget]
    t1  = row["threshold"]

    stage1_anomaly = p_s1 >= t1
    pred = np.zeros(len(y_test_orig), dtype=np.int8)
    if stage1_anomaly.sum() > 0:
        pred[stage1_anomaly] = np.where(p_s2[stage1_anomaly] >= 0.5, 1, 2)

    fp     = ((y_test_orig == 0) & (pred > 0)).sum()
    fpr    = fp / n_normal_test
    atk_r  = (pred[y_test_orig == 1] == 1).mean() if n_attack_test > 0 else 0
    flt_r  = (pred[y_test_orig == 2] == 2).mean() if n_fault_test  > 0 else 0
    cascade_results.append({
        "fpr_budget": f"{budget:.1%}", "threshold": round(t1, 3),
        "attack_recall": round(atk_r, 4), "fault_recall": round(flt_r, 4),
        "normal_fpr": round(fpr, 4),
    })

    # Cache the 1% FPR predictions for reuse in per-fault, confusion matrix, and figure cells
    if abs(budget - 0.010) < 1e-9:
        pred_1pct = pred.copy()
        t1_1pct   = t1

casc_df = pd.DataFrame(cascade_results)
print("Full Cascade Results:")
print(casc_df.to_string(index=False))
casc_df.to_csv(RESULTS_DIR / "cascade_results.csv", index=False)


Full Cascade Results:
fpr_budget  threshold  attack_recall  fault_recall  normal_fpr
      0.5%      0.440         0.9933        0.2663      0.0049
      1.0%      0.320         0.9974        0.4007      0.0099
      2.0%      0.201         0.9995        0.5093      0.0188


## 7. Per-Fault-Type Recall (at 1% FPR Budget)

In [8]:
per_fault_rows = []
if "pred_1pct" in dir() and fault_type_test is not None:
    fault_mask = y_test_orig == 2
    for ft in sorted(set(fault_type_test[fault_mask])):
        if not ft or (isinstance(ft, float) and np.isnan(ft)):
            continue
        ft_mask   = fault_mask & (fault_type_test == ft)
        n_ft      = ft_mask.sum()
        n_correct = (pred_1pct[ft_mask] == 2).sum()
        per_fault_rows.append({
            "fault_type": ft,
            "n_test_rows": int(n_ft),
            "recall": round(n_correct / n_ft, 4) if n_ft > 0 else 0,
        })

    pf_df = pd.DataFrame(per_fault_rows)
    print(f"Per-Fault-Type Recall (FPR budget=1.0%, t1={t1_1pct:.3f}):")
    print(pf_df.to_string(index=False))
    pf_df.to_csv(RESULTS_DIR / "per_fault_recall.csv", index=False)
else:
    print("pred_1pct not available — run cascade evaluation cell first.")


Per-Fault-Type Recall (FPR budget=1.0%, t1=0.320):
           fault_type  n_test_rows  recall
                 bias          376  0.5372
                drift          413  0.2639
 intermittent_dropout          459  0.3987
precision_degradation          547  0.4808
             stuck_at          139  0.1295


## 8. FPR Budget Sensitivity Table

In [9]:
print("FPR Budget Sensitivity:")
print(f"{'FPR Budget':>12}  {'Fault Recall':>13}  {'Attack Recall':>14}  {'Threshold':>10}")
for r in cascade_results:
    print(f"  {r['fpr_budget']:>8}  {r['fault_recall']:>12.1%}  {r['attack_recall']:>13.1%}  {r['threshold']:>10.3f}")


FPR Budget Sensitivity:
  FPR Budget   Fault Recall   Attack Recall   Threshold
      0.5%         26.6%          99.3%       0.440
      1.0%         40.1%          99.7%       0.320
      2.0%         50.9%         100.0%       0.201


## 9. Confusion Matrix (at 1% FPR Budget)

In [10]:
if "pred_1pct" not in dir():
    print("pred_1pct not available — run cascade evaluation cell first.")
else:
    cm = confusion_matrix(y_test_orig, pred_1pct, labels=[0, 1, 2])
    class_names = ["Normal", "Attack", "Fault"]

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    plt.colorbar(im, ax=ax)
    ax.set_xticks([0, 1, 2])
    ax.set_yticks([0, 1, 2])
    ax.set_xticklabels(class_names, fontsize=11)
    ax.set_yticklabels(class_names, fontsize=11)
    ax.set_xlabel("Predicted Label", fontsize=12)
    ax.set_ylabel("True Label", fontsize=12)
    ax.set_title(f"Cascade Confusion Matrix (FPR budget=1.0%, t={t1_1pct:.3f})", fontsize=12)

    thresh_cm = cm.max() / 2.0
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{cm[i,j]:,}", ha="center", va="center",
                    color="white" if cm[i, j] > thresh_cm else "black", fontsize=11)

    fig.tight_layout()
    out = FIGURES_DIR / "confusion_matrix.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out}")
    print(f"\n{classification_report(y_test_orig, pred_1pct, target_names=class_names)}")


Saved: data/results/figures/confusion_matrix.png

              precision    recall  f1-score   support

      Normal       0.99      0.99      0.99    165986
      Attack       0.98      1.00      0.99      1953
       Fault       0.32      0.40      0.36      1934

    accuracy                           0.98    169873
   macro avg       0.77      0.80      0.78    169873
weighted avg       0.99      0.98      0.98    169873



## 10. Stage 1 ROC-Style Fault/Attack Recall vs FPR Curve

In [11]:
fig, ax = plt.subplots(figsize=(7, 5))
fpr_vals   = s1_df["fpr"].values
fault_vals = s1_df["fault_recall"].values
atk_vals   = s1_df["attack_recall"].values

ax.plot(fpr_vals, fault_vals, label="Fault Recall", color="firebrick", lw=1.5)
ax.plot(fpr_vals, atk_vals,   label="Attack Recall", color="steelblue", lw=1.5)

for budget in FPR_BUDGETS:
    ax.axvline(x=budget, color="gray", linestyle="--", lw=0.8, alpha=0.7)
    ax.text(budget + 0.001, 0.05, f"{budget:.1%}", fontsize=8, color="gray")

ax.set_xlabel("Normal FPR", fontsize=12)
ax.set_ylabel("Recall", fontsize=12)
ax.set_title("Stage 1 Recall vs FPR Tradeoff", fontsize=13)
ax.legend(fontsize=10)
ax.set_xlim(0, 0.06)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)

fig.tight_layout()
out = FIGURES_DIR / "stage1_recall_vs_fpr.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out}")


Saved: data/results/figures/stage1_recall_vs_fpr.png


## 11. Save Models & Results

In [12]:
joblib.dump(rf_s1, MODELS_DIR / "stage1_rf.joblib")
joblib.dump(rf_s2, MODELS_DIR / "stage2_rf.joblib")
print(f"Models saved to {MODELS_DIR}/")

results_summary = {
    "stage1_oob_score":   round(rf_s1.oob_score_, 4),
    "stage2_oob_score":   round(rf_s2.oob_score_, 4),
    "stage2_standalone": {
        "attack_recall": round(s2_attack_recall, 4),
        "fault_recall":  round(s2_fault_recall, 4),
        "accuracy":      round(s2_accuracy, 4),
    },
    "cascade_operating_points": cascade_results,
}

(RESULTS_DIR / "final_results.json").write_text(
    json.dumps(results_summary, indent=2)
)
print(f"Saved: {RESULTS_DIR / 'final_results.json'}")
print("\nAll done.")

print(f"Completed: {datetime.now()}")


Models saved to data/models/
Saved: data/results/final_results.json

All done.
Completed: 2026-04-19 17:10:53.986904
